# Ingest: USDA Crop Statistics

Agriculture demo data. Downloads USDA NASS's Census of Agriculture bulk
files (tab-delimited, no auth) and persists a filtered subset as a
managed Spark table.

The Census of Agriculture only runs every 5 years, so a single year has
no prior year to compare against — this pulls **two** census years
(2012 and 2017) so the trend transform downstream has something to
compute a change against.

Confirmed live (2026-08-07) column layout (tab-separated):
`SOURCE_DESC, SECTOR_DESC, ..., COMMODITY_DESC, ..., STATISTICCAT_DESC,
UNIT_DESC, ..., AGG_LEVEL_DESC, ..., STATE_ALPHA, STATE_NAME, ..., YEAR,
..., VALUE, CV_%`. The full file spans every sector (crops, livestock,
economics, demographics) at every geography level (national down to
county/zip) — this ingest filters down to state-level crop
area/production/yield rows to keep the table demo-sized and focused;
`VALUE` is left as the raw source string (comma-formatted numbers,
`(D)`/`(Z)`/`(NA)` suppression sentinels) — cleaned in the transform step,
same pattern as the numeric sentinel handling used elsewhere in this repo.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and
`mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "agriculture"
TABLE_NAME = "crop_statistics"
CENSUS_YEARS = ["2012", "2017"]
SOURCE_URL_TEMPLATE = "https://www.nass.usda.gov/datasets/qs.census{year}.txt.gz"
STATISTIC_CATEGORIES = ["AREA HARVESTED", "PRODUCTION", "YIELD"]

## Download source files

A `User-Agent` header is required — USDA's server behaves inconsistently
without one.

In [ ]:
import urllib.request

local_files = []
for year in CENSUS_YEARS:
    url = SOURCE_URL_TEMPLATE.format(year=year)
    local_file = f"/tmp/qs_census{year}.txt.gz"
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request) as response, open(local_file, "wb") as out_file:
        out_file.write(response.read())
    local_files.append(local_file)

local_files

## Load and union both census years

In [ ]:
raw_df = (
    spark.read
    .option("sep", "\t")
    .option("header", "true")
    .csv(local_files)
)

## Filter to state-level crop area/production/yield rows

In [ ]:
df = (
    raw_df
    .filter(F.col("SOURCE_DESC") == "CENSUS")
    .filter(F.col("SECTOR_DESC") == "CROPS")
    .filter(F.col("AGG_LEVEL_DESC") == "STATE")
    .filter(F.col("STATISTICCAT_DESC").isin(STATISTIC_CATEGORIES))
    .select(
        "COMMODITY_DESC", "STATISTICCAT_DESC", "UNIT_DESC",
        "STATE_ALPHA", "STATE_NAME", "YEAR", "VALUE",
    )
)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'USDA NASS crop statistics used for agriculture demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)